### load patches

In [1]:
import numpy as np, zarr, json, os
from pathlib import Path
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.filters import threshold_otsu

TRAIN_ITER = "v3"
OVERWRITE = True

VIDEOS = [
    "44b6_33b596bf", "44b6_40c45f5a", "44b6_551a5dba", # used in baseline (v0)
    "6bba_61dd1e0d", # added in first iter (v1)

]
MAX_PATCHES = {
    "6bba_61dd1e0d": 100, # added in first iter (v1)
}

raw, lab = {}, {}
for p in VIDEOS:
    r = np.asarray(zarr.open(f'unet_patches/{p}.zarr/0', mode="r"))
    l = np.load(f'unet_labels/{p}_pts.npy', allow_pickle=True)
    n = MAX_PATCHES.get(p)
    if n is not None:
        r = r[:n]
        if p == "background":
            l = np.zeros((0, 4), dtype=np.int32) 
        else:
            l = l[l[:, 0] < n]                         
    raw[p], lab[p] = r, l
    print(p, raw[p].shape, lab[p].shape)

44b6_33b596bf (50, 16, 64, 64) (678, 4)
44b6_40c45f5a (200, 16, 64, 64) (3782, 4)
44b6_551a5dba (228, 16, 64, 64) (6196, 4)
6bba_61dd1e0d (100, 16, 64, 64) (231, 4)


### Gaussian heatmaps

In [6]:
# modified gaussian maps in iter 2
# modified gaussian maps in iter 3 
FLOOR = np.array([1.0, 4.0, 4.0])     # never go below the current fixed sigma
CAP   = np.array([3.0, 12.0, 12.0])   # largest sigma allowed (z, y, x), must fit the patch
FRAC  = 0.75                          # fraction of the watershed radius used as sigma
MAX_DIST = 40                         # flood limit from a seed, in xy-voxel units

SPACING = np.array([4.0, 1.0, 1.0])     # z, y, x voxel size in xy-pixel units
GRAD_SIGMA = (0.5, 1.5, 1.5)            # light smoothing inside the gradient (voxels)
SEED_R = 2.0                            # seed blob radius, xy-pixel units
BG_MIN_DIST = 14.0                      # background marker must be farther than this from any seed
MAX_DIST = 40.0                         # flood limit from a seed, xy-pixel units
zz, yy, xx = np.ogrid[:16, :64, :64]
FRAC_B = 0.5         # sigma = FRAC_B * distance-to-boundary (voxels along each axis)
AGG = "min"          # "min" or "mean" of the two directions

def axis_extent(lab_, j, p, axis):
    """Steps from p along +/- axis while still in region j. Returns [(steps, hit_patch_border), ...] for - and +."""
    out = []
    for sgn in (-1, 1):
        q = list(p); n = 0; border = False
        while True:
            q[axis] += sgn
            if q[axis] < 0 or q[axis] >= lab_.shape[axis]:
                border = True; break
            if lab_[tuple(q)] != j:
                break
            n += 1
        out.append((n + 0.5, border))           # +0.5: boundary lies between voxels
    return out

def boundary_sigmas(lab_, j, p):
    d = np.zeros(3)
    for ax in range(3):
        (dn, bn), (dp, bp) = axis_extent(lab_, j, p, ax)
        valid = [v for v, b in ((dn, bn), (dp, bp)) if not b]     # sides that ended at a true boundary
        if not valid:
            d[ax] = max(dn, dp)                                   # both sides hit the border: lower bound
        elif AGG == "mean":
            d[ax] = np.mean(valid)
        else:
            d[ax] = min(valid)
    return d

def pts_for_patch(arr, i):
    if arr.dtype == object:                          # one array of points per patch
        return np.asarray(arr[i], float).reshape(-1, 3)
    if arr.ndim == 2 and arr.shape[1] == 4:          # rows of (patch_idx, z, y, x)
        return arr[arr[:, 0] == i, 1:].astype(float)
    raise ValueError(f"unrecognised points layout: {arr.shape}")

def grad_mag(im):
    """Gradient magnitude in physical units (z derivative divided by the z spacing)."""
    gz = ndi.gaussian_filter(im, GRAD_SIGMA, order=(1, 0, 0)) / SPACING[0]
    gy = ndi.gaussian_filter(im, GRAD_SIGMA, order=(0, 1, 0))
    gx = ndi.gaussian_filter(im, GRAD_SIGMA, order=(0, 0, 1))
    return np.sqrt(gz**2 + gy**2 + gx**2)


In [7]:
def sigmas_for_patch(img, pts, return_mask=False):
    """Per-point (sz, sy, sx) from a seeded watershed on the gradient magnitude. Falls back to FLOOR."""
    k = len(pts)
    sig = np.tile(FLOOR, (k, 1))
    empty = np.zeros((16, 64, 64), np.int16)
    if k == 0:
        return (sig, empty) if return_mask else sig
    img = img.astype(np.float32)
    rng = img.max() - img.min()
    if rng < 1e-6:
        return (sig, empty) if return_mask else sig
    im = (img - img.min()) / rng
    ims = ndi.gaussian_filter(im, GRAD_SIGMA)
    g = grad_mag(im)

    # seeds: small blobs around each annotated point (nearest seed wins)
    single = np.zeros(im.shape, bool)
    for (z, y, x) in pts:
        single[int(np.clip(round(z), 0, 15)), int(np.clip(round(y), 0, 63)), int(np.clip(round(x), 0, 63))] = True
    idx_label = np.zeros(im.shape, np.int32)
    for j, (z, y, x) in enumerate(pts, 1):
        idx_label[int(np.clip(round(z), 0, 15)), int(np.clip(round(y), 0, 63)), int(np.clip(round(x), 0, 63))] = j
    dist, (iz, iy, ix) = ndi.distance_transform_edt(~single, sampling=SPACING, return_indices=True)
    nearest = idx_label[iz, iy, ix]

    markers = np.where(dist <= SEED_R, nearest, 0).astype(np.int32)
    # background marker: dark AND far from every seed (skipped if no such voxels)
    bg = (ims < threshold_otsu(ims)) & (dist > BG_MIN_DIST)
    if bg.any():
        markers[bg] = k + 1

    lab_ = watershed(g, markers, mask=dist <= MAX_DIST)     # compactness=0 (voxels are anisotropic)

    for j, (z, y, x) in enumerate(pts, 1):
        p = (int(np.clip(round(z), 0, 15)), int(np.clip(round(y), 0, 63)), int(np.clip(round(x), 0, 63)))
        if lab_[p] != j:                      # seed lost to a neighbour or the background: keep the floor
            continue
        sig[j - 1] = np.clip(FRAC_B * boundary_sigmas(lab_, j, p), FLOOR, CAP)

    if return_mask:
        out = lab_.copy(); out[out > k] = 0      # background label -> 0
        return sig, out.astype(np.int16)
    
    return sig

def make_heatmap(pts, sigmas):
    hm = np.zeros((16, 64, 64), np.float32)
    for (cz, cy, cx), (sz, sy, sx) in zip(pts, sigmas):
        g = np.exp(-(((zz - cz) / sz) ** 2 + ((yy - cy) / sy) ** 2 + ((xx - cx) / sx) ** 2) / 2)
        hm = np.maximum(hm, g.astype(np.float32))
    return hm


In [8]:
heat, all_sig = {}, {}
for name in VIDEOS:
    out = Path("heatmaps") / f"{name}_{TRAIN_ITER}_heatmap.npy"
    sig_out = Path("heatmaps") / f"{name}_{TRAIN_ITER}_sigmas.npy"
    out_ws = Path("heatmaps") / f"{name}_{TRAIN_ITER}_watersheds.npy"

    if not OVERWRITE and out.exists():
        heat[name] = np.load(out); print("loaded existing:", out); continue
  
    R = raw[name].squeeze() if raw[name].ndim == 5 else raw[name]
    N = R.shape[0]

    hms, sgs, wss = [], [], []
    for i in range(N):
        pts = pts_for_patch(lab[name], i)
        s, ws = sigmas_for_patch(R[i], pts, return_mask=True)
        hms.append(make_heatmap(pts, s)); sgs.append(s); wss.append(ws)
    
    heat[name] = np.stack(hms)
    all_sig[name] = np.concatenate(sgs) if sgs else np.zeros((0, 3))
    np.save(out_ws, np.stack(wss))      # (N,16,64,64) int16
    np.save(sig_out, all_sig[name])
    np.save(out, heat[name])

    s = all_sig[name]
    print(name, "generated", heat[name].shape,
          "| patches with no points:", int((heat[name].reshape(N, -1).max(1) == 0).sum()))
    print("   sigma median (z,y,x):", np.median(s, 0).round(2),
          "| 95th pct:", np.percentile(s, 95, axis=0).round(2),
          "| at floor:", f"{(np.all(s <= FLOOR + 1e-6, axis=1)).mean():.0%}",
          "| at cap:", f"{(np.any(s >= CAP - 1e-6, axis=1)).mean():.0%}")

44b6_33b596bf generated (50, 16, 64, 64) | patches with no points: 0
   sigma median (z,y,x): [1.25 4.   4.  ] | 95th pct: [2.25 4.   4.  ] | at floor: 19% | at cap: 2%
44b6_40c45f5a generated (200, 16, 64, 64) | patches with no points: 0
   sigma median (z,y,x): [1.75 4.   4.  ] | 95th pct: [2.75 4.   4.25] | at floor: 16% | at cap: 3%
44b6_551a5dba generated (228, 16, 64, 64) | patches with no points: 0
   sigma median (z,y,x): [1.25 4.   4.  ] | 95th pct: [2.75 4.   4.25] | at floor: 23% | at cap: 5%
6bba_61dd1e0d generated (100, 16, 64, 64) | patches with no points: 0
   sigma median (z,y,x): [1.75 4.   4.  ] | 95th pct: [2.25 5.75 6.25] | at floor: 23% | at cap: 1%


### preprocessing

In [9]:
import torch
from torch.utils.data import TensorDataset, DataLoader

norm_path = f"models/norm_{TRAIN_ITER}.json"

if not OVERWRITE and os.path.exists(norm_path):
    with open(norm_path, "r") as f:
        norm_params = json.load(f)
    lo, hi = norm_params["lo"], norm_params["hi"]
    print("loaded existing norm:", norm_path)
else:
    allraw = np.concatenate([raw[n].ravel() for n in VIDEOS])
    lo, hi = np.percentile(allraw, [1, 99.8])

    with open(norm_path, "w") as f:
        json.dump({"lo": float(lo), "hi": float(hi)}, f)

    print("computed and saved norm:", norm_path)

print("norm range:", lo, hi)

def norm(x):
    return np.clip((x - lo) / (hi - lo + 1e-8), 0, 1).astype(np.float32)

X = np.concatenate([norm(raw[n]) for n in VIDEOS])[:, None]
Y = np.concatenate([heat[n] for n in VIDEOS])[:, None]

print(X.shape, Y.shape, "target max:", Y.max())

loader = DataLoader(
    TensorDataset(torch.from_numpy(X), torch.from_numpy(Y)),
    batch_size=8,
    shuffle=True,
    drop_last=False
)

computed and saved norm: models/norm_v3.json
norm range: 0.0 3614.0
(578, 1, 16, 64, 64) (578, 1, 16, 64, 64) target max: 1.0


### 3D U-Net

In [10]:
%%writefile unet3d.py
import torch, torch.nn as nn

def block(i, o):
    return nn.Sequential(
        nn.Conv3d(i, o, 3, padding=1), nn.BatchNorm3d(o), nn.ReLU(inplace=True),
        nn.Conv3d(o, o, 3, padding=1), nn.BatchNorm3d(o), nn.ReLU(inplace=True)
        )

class UNet3D(nn.Module):
    # anisotropic pooling: z is only 16 slices, so don't squash it early
    def __init__(self, c=16):
        super().__init__()
        self.e1, self.e2, self.e3, self.b = block(1, c), block(c, 2*c), block(2*c, 4*c), block(4*c, 8*c)
        self.p1, self.p2, self.p3 = nn.MaxPool3d((1,2,2)), nn.MaxPool3d(2), nn.MaxPool3d(2)
        self.u3 = nn.ConvTranspose3d(8*c, 4*c, 2, stride=2)
        self.u2 = nn.ConvTranspose3d(4*c, 2*c, 2, stride=2)
        self.u1 = nn.ConvTranspose3d(2*c, c, (1,2,2), stride=(1,2,2))
        self.d3, self.d2, self.d1 = block(8*c, 4*c), block(4*c, 2*c), block(2*c, c)
        self.out = nn.Conv3d(c, 1, 1)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p1(e1))
        e3 = self.e3(self.p2(e2))
        b = self.b(self.p3(e3))
        d3 = self.d3(torch.cat([self.u3(b), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return torch.sigmoid(self.out(d1))

Overwriting unet3d.py


In [11]:
from unet3d import UNet3D
device = "cuda" if torch.cuda.is_available() else "cpu"
model = UNet3D(16).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print(device, sum(p.numel() for p in model.parameters())/1e6, "M params")

cuda 1.399921 M params


### training

In [12]:
EPOCHS, SAVE_EVERY = 100, 10

def weighted_mse(pred, target):
    w = 1 + 20 * target          # upweight cell voxels, background is the vast majority
    return (w * (pred - target) ** 2).mean()

for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)   
        opt.zero_grad()
        loss = weighted_mse(model(xb), yb)
        loss.backward(); opt.step()
        tot += loss.item() * len(xb)
    print(f"epoch {ep:3d}  loss {tot/len(X):.5f}")
    if ep % SAVE_EVERY == 0:
        torch.save(model.state_dict(), f"models/unet_{TRAIN_ITER}_e{ep}.pt")

epoch   1  loss 0.21649
epoch   2  loss 0.11179
epoch   3  loss 0.06986
epoch   4  loss 0.05187
epoch   5  loss 0.04082
epoch   6  loss 0.03450
epoch   7  loss 0.03063
epoch   8  loss 0.02740
epoch   9  loss 0.02527
epoch  10  loss 0.02386
epoch  11  loss 0.02211
epoch  12  loss 0.02127
epoch  13  loss 0.01990
epoch  14  loss 0.01891
epoch  15  loss 0.01823
epoch  16  loss 0.01678
epoch  17  loss 0.01648
epoch  18  loss 0.01576
epoch  19  loss 0.01537
epoch  20  loss 0.01361
epoch  21  loss 0.01243
epoch  22  loss 0.01181
epoch  23  loss 0.01099
epoch  24  loss 0.01055
epoch  25  loss 0.00969
epoch  26  loss 0.01050
epoch  27  loss 0.00935
epoch  28  loss 0.00816
epoch  29  loss 0.00767
epoch  30  loss 0.00757
epoch  31  loss 0.00692
epoch  32  loss 0.00634
epoch  33  loss 0.00769
epoch  34  loss 0.00668
epoch  35  loss 0.00599
epoch  36  loss 0.00538
epoch  37  loss 0.00451
epoch  38  loss 0.00425
epoch  39  loss 0.00400
epoch  40  loss 0.00447
epoch  41  loss 0.00423
epoch  42  loss 